# SWAP-Stress: validation (stage 04)

How well the released model does on data it never saw. The holdout is
**spatial**: whole 9 km pixels are held out, not random rows, so a site's own
observations cannot leak between train and test.

1. The held-out result
2. Predicted against observed, by source
3. Residual diagnostics
4. The held-out set with its coordinates
5. Per-site skill in space
6. The rest of stage 04

Metrics come from `swapstress.model.metrics` — the same functions that wrote the
model's own artifacts — so nothing here can quietly disagree with the numbers in
the descriptor.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from swapstress.config import load_config

# ---------------------------------------------------------------------------
# The one thing you may need to change: where this repo is checked out. The
# model directory is whatever the train config wrote to.
# ---------------------------------------------------------------------------
REPO = os.path.abspath(os.environ.get("SWAPSTRESS_REPO", "."))

train_cfg = load_config(
    os.path.join(REPO, "configs", "train_9km_global_pruned.toml"), {}
)
MODEL_DIR = train_cfg["output_dir"]

# Reconstructing the holdout re-runs the split and re-predicts on first call
# (minutes); afterwards it loads a validated cache from the model directory.
RUN_RECONSTRUCT = False

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Display plumbing: one stable colour per source across every panel.
SOURCE_COLORS = {
    "gshp": "#4e79a7",
    "ncss": "#76b7b2",
    "mt_mesonet": "#f28e2b",
    "reesh": "#e15759",
    "lacadian": "#59a14f",
}
DEFAULT_COLOR = "#9c755f"

print("MODEL_DIR:  ", MODEL_DIR)
print("obs table:  ", train_cfg["obs_table"])
print("kept groups:", ", ".join(train_cfg["feature_groups"]))
print(
    "holdout:    ",
    f"{train_cfg['test_size']:.0%} of "
    f"{train_cfg['resolution_m']} m spatial groups, seed {train_cfg['random_state']}",
)

## 1) The held-out result

`swapstress.validation.baseline_summary` is the `baseline` analysis of stage 04.
`load_artifacts` reads the four things a trained model directory carries —
`direct_model_results.json`, `predictions.parquet`, and the per-source and
per-site metric tables — and `format_summary` renders the same text block the
stage writes to `baseline_summary.txt`.

In [ ]:
from swapstress.validation.baseline_summary import format_summary, load_artifacts

artifacts = load_artifacts(MODEL_DIR)
print(format_summary(artifacts))

In [ ]:
pred_df = artifacts["predictions"]
results = artifacts["results"]

print(
    f"predictions: {pred_df.shape[0]:,} rows x {pred_df.shape[1]} cols "
    f"{list(pred_df.columns)}"
)

overall = results["overall_metrics"]
print("\noverall (log10 cm):")
for k in ["r2", "rmse", "mae", "bias", "n"]:
    v = overall[k]
    print(f"  {k:<5s} {v:,.4f}" if isinstance(v, float) else f"  {k:<5s} {v:,}")

if artifacts["source_metrics"] is not None:
    print()
    print(artifacts["source_metrics"].to_string(index=False))

## 2) Predicted against observed, by source

One panel per training source, the dashed line is 1:1. The panels are not
comparable as skill scores: each source spans a different range of suction, and
R² against a narrow range is a harder test than the same error against a wide
one. The lab sources (GSHP, NCSS) cover the full retention curve; the in-situ
sensors are bounded by what a tensiometer can read.

In [ ]:
from swapstress.model.metrics import compute_metrics_by_source

by_source = compute_metrics_by_source(
    pred_df["observed"].to_numpy(),
    pred_df["predicted"].to_numpy(),
    pred_df["source"].to_numpy(),
)
by_source

In [ ]:
sources = sorted(pred_df["source"].dropna().unique())
metrics_lookup = by_source.set_index("source")

ncols = min(3, len(sources))
nrows = (len(sources) + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False, dpi=120
)

lo, hi = np.percentile(
    pd.concat([pred_df["observed"], pred_df["predicted"]]).dropna(), [1, 99]
)

for idx, src in enumerate(sources):
    ax = axes[idx // ncols][idx % ncols]
    sub = pred_df[pred_df["source"] == src].dropna(subset=["observed", "predicted"])
    ax.scatter(
        sub["observed"],
        sub["predicted"],
        s=6,
        alpha=0.3,
        color=SOURCE_COLORS.get(src, DEFAULT_COLOR),
        rasterized=True,
    )
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, alpha=0.6)

    m = metrics_lookup.loc[src]
    ax.text(
        0.04,
        0.94,
        f"R$^2$={m['r2']:.2f}  RMSE={m['rmse']:.3f}",
        transform=ax.transAxes,
        fontsize=8,
        va="top",
    )
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Observed log10 suction (cm)", fontsize=8)
    ax.set_ylabel("Predicted", fontsize=8)
    ax.set_title(f"{src}  (n={len(sub):,})", fontsize=10)
    ax.tick_params(labelsize=7)

for idx in range(len(sources), nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle("Predicted against observed log10 suction, by source", fontsize=13)
fig.tight_layout()
out = os.path.join(OUT_DIR, "08_scatter_by_source.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 3) Residual diagnostics

Residual = predicted − observed. Two things to look for: whether the spread
widens with observed suction (heteroscedasticity, which the empirical error
lookup in stage 04 quantifies), and whether any source's residuals are centred
off zero (a source-specific bias, which the conditional-bias analysis pursues).

In [ ]:
from scipy.stats import gaussian_kde

from swapstress.model.metrics import compute_metrics

pred_df = pred_df.assign(residual=pred_df["predicted"] - pred_df["observed"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

for src in sources:
    sub = pred_df[pred_df["source"] == src].dropna(subset=["observed", "residual"])
    color = SOURCE_COLORS.get(src, DEFAULT_COLOR)
    ax1.scatter(
        sub["observed"],
        sub["residual"],
        s=4,
        alpha=0.2,
        color=color,
        label=src,
        rasterized=True,
    )
    kde = gaussian_kde(sub["residual"])
    grid = np.linspace(sub["residual"].min(), sub["residual"].max(), 300)
    ax2.plot(grid, kde(grid), color=color, lw=1.5, label=src)

ax1.axhline(0, color="k", lw=0.8, ls="--")
ax1.set_xlabel("Observed log10 suction (cm)")
ax1.set_ylabel("Residual (predicted − observed)")
ax1.set_title("Residuals against observed")
ax1.legend(markerscale=3, fontsize=8)

ax2.axvline(0, color="k", lw=0.8, ls="--")
ax2.set_xlabel("Residual (predicted − observed)")
ax2.set_ylabel("Density")
ax2.set_title("Residual distribution per source")
ax2.legend(fontsize=8)

fig.suptitle("Residual diagnostics", fontsize=13)
fig.tight_layout()
out = os.path.join(OUT_DIR, "08_residuals.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

print("\nresiduals by observed decile:")
deciles = pd.qcut(pred_df["observed"], 10, duplicates="drop")
for interval, g in pred_df.groupby(deciles, observed=True):
    m = compute_metrics(g["observed"].to_numpy(), g["predicted"].to_numpy())
    print(
        f"  {str(interval):<18s} n={m['n']:>7,}  bias={m['bias']:+.3f}  "
        f"rmse={m['rmse']:.3f}"
    )

## 4) The held-out set with its coordinates

`predictions.parquet` carries only `observed`, `predicted`, and `source` — no
coordinates — so a spatial view needs the split reproduced.
`swapstress.validation.reconstruct_test_set.reconstruct` does exactly that: it
reads `resolution_m` back out of the model artifacts, re-runs
`prepare_direct_data` with the saved config, re-predicts with the saved model and
imputer, and checks the resulting R² against the saved one before returning.

The result is cached to `<model_dir>/test_set_full.parquet` and revalidated (row
count and R²) on every later call, so a stale cache rebuilds rather than
silently misleading.

In [ ]:
test_df = None

if RUN_RECONSTRUCT:
    from swapstress.validation.reconstruct_test_set import reconstruct

    test_df = reconstruct(MODEL_DIR)
    print(f"\n{len(test_df):,} rows x {test_df.shape[1]} columns")
    print(
        "identity columns:",
        [
            c
            for c in [
                "source",
                "sample_id",
                "lat",
                "lon",
                "theta",
                "depth_cm",
                "observed",
                "predicted",
            ]
            if c in test_df.columns
        ],
    )
else:
    print("RUN_RECONSTRUCT=False — sections 4 and 5 need it")

## 5) Per-site skill in space

`assign_spatial_group` is the same 9 km binning the split used, so a "site" here
is exactly a holdout unit. `compute_metrics_by_site` drops sites with fewer than
three observations, because RMSE over two points is noise.

Where skill collapses is the useful signal: it marks either a climate or soil
regime the training data does not cover, or a source whose measurement support
does not match the 9 km pixel. Stage 04's `regional` and `distribution-shift`
analyses are the systematic versions of this glance.

In [ ]:
if test_df is not None:
    from swapstress.figures import basemap
    from swapstress.model.data import assign_spatial_group
    from swapstress.model.metrics import compute_metrics_by_site

    groups = assign_spatial_group(test_df, resolution_m=train_cfg["resolution_m"])
    per_site, summary = compute_metrics_by_site(
        test_df["observed"].to_numpy(),
        test_df["predicted"].to_numpy(),
        groups.to_numpy(),
    )

    coords = (
        test_df.assign(site_id=groups.to_numpy())
        .groupby("site_id")[["lat", "lon"]]
        .mean()
    )
    per_site = per_site.merge(coords, left_on="site_id", right_index=True)

    print(f"sites with >=3 held-out observations: {summary['n_sites']:,}")
    print(
        f"median site RMSE: {summary['median_rmse']:.3f} log10 cm  "
        f"(mean {summary['mean_rmse']:.3f})"
    )
    print(f"median site R2:   {summary['median_r2']:.3f}")

    fig, ax = plt.subplots(figsize=(13, 7), dpi=130)
    # Display plumbing: the descriptor's map figures project to Albers and style
    # their own axes, so a notebook glance corrects the aspect by latitude.
    basemap.load_conus_states().boundary.plot(ax=ax, color="0.5", linewidth=0.4)
    ax.set_xlim(*basemap.CONUS_LON)
    ax.set_ylim(*basemap.CONUS_LAT)
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(basemap.CONUS_LAT))))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    sc = ax.scatter(
        per_site["lon"],
        per_site["lat"],
        c=per_site["rmse"],
        cmap="RdYlGn_r",
        s=22,
        alpha=0.8,
        edgecolors="none",
        vmin=np.percentile(per_site["rmse"], 2),
        vmax=np.percentile(per_site["rmse"], 98),
    )
    fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.02, label="RMSE (log10 suction cm)")
    ax.set_title("Per-site RMSE on the spatial holdout")

    fig.tight_layout()
    out = os.path.join(OUT_DIR, "08_spatial_rmse.png")
    fig.savefig(out, dpi=200)
    plt.show()
    print("Saved:", os.path.abspath(out))
else:
    print("section 4 not run — skipping the spatial map")

In [ ]:
if test_df is not None:
    worst = per_site.sort_values("rmse", ascending=False).head(10)
    best = per_site.sort_values("rmse").head(10)
    print("worst 10 sites:")
    cols = ["site_id", "n", "rmse", "r2", "bias", "lat", "lon"]
    print(worst[cols].to_string(index=False))
    print("\nbest 10 sites:")
    print(best[cols].to_string(index=False))

## 6) The rest of stage 04

The `baseline` analysis above is one of eight. The others ask different
questions, need different inputs, and each writes its own outputs under
`<model-dir>/error_analysis`:

```bash
uv run swapstress-validate --model-dir <model dir> --analysis all
```

`--analysis all` deliberately excludes `ptf-baseline`: it has its own two-step
prep/eval interface and reads an external Rosetta grid, so it is asked for by
name.

In [ ]:
from swapstress.validation import run as validation_run

for name, module in validation_run.ANALYSES.items():
    mark = " " if name in validation_run.DEFAULT_ANALYSES else "*"
    print(f" {mark} {name:<20s} {module}")
print("\n* not included in --analysis all")

print()
validation_run.main(["--model-dir", MODEL_DIR, "--dry-run"])

## Next

Stage 08 renders the descriptor figures from these same artifacts:

```bash
uv run swapstress-figures --figure all
```

`swapstress.figures.run.MAIN_FIGURES` lists them; `fig05_ptf_comparison` is
the published form of section 2 above.